# Predicting Irrigation Need — Exploratory Data Analysis

Kaggle Playground Series S6E4. Target: `Irrigation_Need` (`Low`/`Medium`/`High`),
metric: **balanced accuracy**.

This notebook answers eight questions:
1. Target distribution
2. Which features have missing values
3. Which "missing" values actually mean *None* rather than absent data
4. Which numerical features are skewed
5. Which categorical features have high cardinality
6. Which features relate most strongly to the target (`Irrigation_Need`; the original
   template asked about `SalePrice`, which does not exist here)
7. Obvious outliers
8. Whether train and test distributions are similar

Figures are written to `../reports/figures/`.

In [1]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid")

RAW = Path("../data/raw")
FIG = Path("../reports/figures"); FIG.mkdir(parents=True, exist_ok=True)
TARGET = "Irrigation_Need"
CLASS_ORDER = ["Low", "Medium", "High"]
SEED = 42

train = pd.read_csv(RAW / "train.csv")
test = pd.read_csv(RAW / "test.csv")
features = [c for c in train.columns if c not in ("id", TARGET)]
num_cols = [c for c in features if pd.api.types.is_numeric_dtype(train[c])]
cat_cols = [c for c in features if c not in num_cols]
print("train:", train.shape, "| test:", test.shape)
print("numeric (%d):" % len(num_cols), num_cols)
print("categorical (%d):" % len(cat_cols), cat_cols)

train: (630000, 21) | test: (270000, 20)
numeric (11): ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']
categorical (8): ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']


## Q1. Target distribution

In [2]:
dist = train[TARGET].value_counts().reindex(CLASS_ORDER)
pct = (dist / len(train) * 100).round(2)
print(pd.DataFrame({"count": dist, "pct": pct}))
print("imbalance ratio (max/min): %.1fx" % (dist.max() / dist.min()))

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=CLASS_ORDER, y=dist.values, ax=ax, palette="viridis")
for i, v in enumerate(dist.values):
    ax.text(i, v, f"{v:,}\n{pct.values[i]}%", ha="center", va="bottom")
ax.set(title="Target distribution: Irrigation_Need", ylabel="count")
fig.tight_layout(); fig.savefig(FIG / "q1_target_distribution.png", dpi=110); plt.close(fig)

                  count    pct
Irrigation_Need               
Low              369917  58.72
Medium           239074  37.95
High              21009   3.33
imbalance ratio (max/min): 17.6x


## Q2. Which features have missing values?

In [3]:
miss = pd.DataFrame({
    "train_missing": train[features].isna().sum(),
    "test_missing": test[features].isna().sum(),
})
miss["train_pct"] = (miss["train_missing"] / len(train) * 100).round(3)
print("Total missing -> train:", int(miss.train_missing.sum()),
      "| test:", int(miss.test_missing.sum()))
print(miss[miss.train_missing + miss.test_missing > 0] if miss.values.sum()
      else "No missing values in any feature (train or test).")

Total missing -> train: 0 | test: 0
No missing values in any feature (train or test).


## Q3. Which "missing" values mean *None* rather than missing data?

There are no `NaN`s, but some categorical levels encode a *real absence* of a process
rather than unknown data — these are semantic "None" values, not gaps to impute.

In [4]:
semantic_none = {
    "Irrigation_Type": ["Rainfed"],   # 'Rainfed' = no active irrigation applied
    "Mulching_Used":   ["No"],         # 'No' = mulching practice absent
}
for col, vals in semantic_none.items():
    sub = train[train[col].isin(vals)]
    share = len(sub) / len(train) * 100
    print(f"{col} in {vals}: {len(sub):,} rows ({share:.1f}%) -- means 'none applied', not missing")
print()
near0 = (train["Previous_Irrigation_mm"] < 1).mean() * 100
print(f"Previous_Irrigation_mm < 1mm: {near0:.2f}% of rows (effectively 'no prior irrigation')")

Irrigation_Type in ['Rainfed']: 155,607 rows (24.7%) -- means 'none applied', not missing
Mulching_Used in ['No']: 316,453 rows (50.2%) -- means 'none applied', not missing

Previous_Irrigation_mm < 1mm: 0.54% of rows (effectively 'no prior irrigation')


## Q4. Which numerical features are skewed?

In [5]:
skew = train[num_cols].apply(lambda s: stats.skew(s.dropna())).sort_values(key=np.abs, ascending=False)
sk = skew.to_frame("skew")
sk["flag"] = np.where(sk["skew"].abs() > 1, "high", np.where(sk["skew"].abs() > 0.5, "moderate", "~symmetric"))
print(sk.round(3))

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, c in zip(axes.ravel(), num_cols):
    sns.histplot(train[c], bins=50, ax=ax, color="steelblue")
    ax.set_title(f"{c} (skew={skew[c]:.2f})", fontsize=9)
for ax in axes.ravel()[len(num_cols):]:
    ax.axis("off")
fig.suptitle("Numeric feature distributions", y=1.01)
fig.tight_layout(); fig.savefig(FIG / "q4_numeric_distributions.png", dpi=110, bbox_inches="tight"); plt.close(fig)

                          skew        flag
Rainfall_mm             -0.118  ~symmetric
Organic_Carbon           0.105  ~symmetric
Humidity                -0.089  ~symmetric
Soil_pH                  0.070  ~symmetric
Soil_Moisture           -0.064  ~symmetric
Field_Area_hectare       0.052  ~symmetric
Electrical_Conductivity  0.048  ~symmetric
Sunlight_Hours          -0.035  ~symmetric
Wind_Speed_kmh          -0.028  ~symmetric
Previous_Irrigation_mm  -0.017  ~symmetric
Temperature_C           -0.004  ~symmetric


## Q5. Which categorical features have high cardinality?

In [6]:
card = pd.Series({c: train[c].nunique() for c in cat_cols}).sort_values(ascending=False)
print(card.to_frame("n_unique"))
print("\nMax cardinality:", card.max(), "-> all categoricals are LOW cardinality.")
print("No high-cardinality encoding (hashing/target-encoding) is required.")

                   n_unique
Crop_Type                 6
Region                    5
Soil_Type                 4
Crop_Growth_Stage         4
Irrigation_Type           4
Water_Source              4
Season                    3
Mulching_Used             2

Max cardinality: 6 -> all categoricals are LOW cardinality.
No high-cardinality encoding (hashing/target-encoding) is required.


## Q6. Which features relate most strongly to the target?

(The template's `SalePrice` does not exist; the target here is the categorical
`Irrigation_Need`.) For a classification target we use **mutual information** plus
per-class group means.

In [7]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

samp = train.sample(min(80000, len(train)), random_state=SEED)
X = samp[features].copy()
for c in cat_cols:
    X[c] = LabelEncoder().fit_transform(X[c].astype(str))
y = LabelEncoder().fit_transform(samp[TARGET])
discrete = [c in cat_cols for c in features]
mi = pd.Series(
    mutual_info_classif(X, y, discrete_features=discrete, random_state=SEED),
    index=features,
).sort_values(ascending=False)
print("Mutual information with Irrigation_Need (80k sample):")
print(mi.round(5).to_frame("mutual_info"))

fig, ax = plt.subplots(figsize=(7, 6))
sns.barplot(x=mi.values, y=mi.index, ax=ax, palette="magma")
ax.set(title="Mutual information with Irrigation_Need", xlabel="MI")
fig.tight_layout(); fig.savefig(FIG / "q6_mutual_information.png", dpi=110); plt.close(fig)

Mutual information with Irrigation_Need (80k sample):
                         mutual_info
Soil_Moisture                0.20368
Rainfall_mm                  0.18856
Crop_Growth_Stage            0.16555
Temperature_C                0.07363
Wind_Speed_kmh               0.06415
Previous_Irrigation_mm       0.04948
Humidity                     0.04859
Mulching_Used                0.04724
Field_Area_hectare           0.01323
Organic_Carbon               0.01019
Electrical_Conductivity      0.01015
Soil_pH                      0.00843
Sunlight_Hours               0.00814
Water_Source                 0.00125
Irrigation_Type              0.00087
Crop_Type                    0.00086
Season                       0.00049
Soil_Type                    0.00040
Region                       0.00016


In [8]:
top_num = [c for c in mi.index if c in num_cols][:5]
print("Per-class means for top numeric features:")
print(train.groupby(TARGET)[top_num].mean().reindex(CLASS_ORDER).round(2).T)

Per-class means for top numeric features:
Irrigation_Need             Low   Medium    High
Soil_Moisture             43.31    29.74   17.67
Rainfall_mm             1500.53  1444.48  989.16
Temperature_C             25.35    28.89   34.57
Wind_Speed_kmh             9.22    11.79   14.64
Previous_Irrigation_mm    61.72    63.18   63.05


## Q7. Are there obvious outliers?

In [9]:
rows = []
for c in num_cols:
    q1, q3 = train[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((train[c] < lo) | (train[c] > hi)).sum()
    rows.append((c, train[c].min(), train[c].max(), round(lo, 2), round(hi, 2),
                 int(n_out), round(n_out / len(train) * 100, 3)))
out = pd.DataFrame(rows, columns=["feature", "min", "max", "iqr_lo", "iqr_hi",
                                  "n_outliers", "pct_outliers"]).sort_values("pct_outliers", ascending=False)
print(out.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 5))
znorm = (train[num_cols] - train[num_cols].mean()) / train[num_cols].std()
sns.boxplot(data=znorm, ax=ax, orient="h", palette="coolwarm")
ax.set(title="Standardized numeric features (boxplots)", xlabel="z-score")
fig.tight_layout(); fig.savefig(FIG / "q7_outlier_boxplots.png", dpi=110); plt.close(fig)

                feature   min     max  iqr_lo  iqr_hi  n_outliers  pct_outliers
                Soil_pH  4.80    8.20    3.32    9.64           0           0.0
          Soil_Moisture  8.00   64.99  -18.56   93.17           0           0.0
         Organic_Carbon  0.30    1.60   -0.31    2.13           0           0.0
Electrical_Conductivity  0.10    3.50   -1.54    5.05           0           0.0
          Temperature_C 12.00   42.00   -3.02   57.07           0           0.0
               Humidity 25.00   94.99   -5.21  129.72           0           0.0
            Rainfall_mm  0.38 2499.69 -695.00 3703.85           0           0.0
         Sunlight_Hours  4.00   11.00    0.52   14.48           0           0.0
         Wind_Speed_kmh  0.50   20.00   -9.94   30.65           0           0.0
     Field_Area_hectare  0.30   15.00   -7.01   22.03           0           0.0
 Previous_Irrigation_mm  0.02  119.99  -56.18  182.01           0           0.0


## Q8. Are train and test distributions similar?

In [10]:
print("Numeric: two-sample KS test (train vs test)")
ks_rows = []
for c in num_cols:
    s, p = stats.ks_2samp(train[c], test[c])
    ks_rows.append((c, round(s, 4), round(p, 4),
                    round(train[c].mean(), 3), round(test[c].mean(), 3)))
ks = pd.DataFrame(ks_rows, columns=["feature", "ks_stat", "p_value", "train_mean", "test_mean"])
ks = ks.sort_values("ks_stat", ascending=False)
print(ks.to_string(index=False))
print("Max KS stat: %.4f (values near 0 => near-identical distributions)" % ks.ks_stat.max())

print("\nCategorical: max abs proportion gap (train vs test)")
for c in cat_cols:
    a = train[c].value_counts(normalize=True)
    b = test[c].value_counts(normalize=True)
    gap = (a - b).abs().max()
    print(f"  {c}: max |dproportion| = {gap:.4f}")

Numeric: two-sample KS test (train vs test)


                feature  ks_stat  p_value  train_mean  test_mean
            Rainfall_mm   0.0027   0.1203    1462.208   1464.526
     Field_Area_hectare   0.0025   0.2050       7.518      7.508
               Humidity   0.0024   0.2135      61.563     61.511
          Temperature_C   0.0020   0.4186      26.998     27.002
         Wind_Speed_kmh   0.0020   0.4438      10.375     10.387
Electrical_Conductivity   0.0019   0.5040       1.745      1.745
         Sunlight_Hours   0.0018   0.5634       7.513      7.513
 Previous_Irrigation_mm   0.0018   0.5420      62.318     62.356
         Organic_Carbon   0.0017   0.6632       0.923      0.922
          Soil_Moisture   0.0015   0.8013      37.304     37.308
                Soil_pH   0.0013   0.9224       6.482      6.481
Max KS stat: 0.0027 (values near 0 => near-identical distributions)

Categorical: max abs proportion gap (train vs test)
  Soil_Type: max |dproportion| = 0.0008
  Crop_Type: max |dproportion| = 0.0021
  Crop_Growth_Stage

  Irrigation_Type: max |dproportion| = 0.0015
  Water_Source: max |dproportion| = 0.0012
  Mulching_Used: max |dproportion| = 0.0021
  Region: max |dproportion| = 0.0005


In [11]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import OrdinalEncoder

adv = pd.concat([train[features].sample(60000, random_state=SEED).assign(_is_test=0),
                 test[features].sample(60000, random_state=SEED).assign(_is_test=1)], ignore_index=True)
Xa = adv[features].copy()
Xa[cat_cols] = OrdinalEncoder().fit_transform(Xa[cat_cols].astype(str))
auc = cross_val_score(HistGradientBoostingClassifier(random_state=SEED, max_iter=150),
                      Xa, adv["_is_test"], cv=3, scoring="roc_auc").mean()
print("Adversarial validation AUC: %.4f  (~0.5 => train and test are indistinguishable)" % auc)

Adversarial validation AUC: 0.4987  (~0.5 => train and test are indistinguishable)
